In [1]:
!pip -q install diffusers transformers accelerate peft datasets
!git clone https://github.com/huggingface/diffusers
%cd diffusers/examples/text_to_image
from huggingface_hub import login
from google.colab import userdata
token = userdata.get('hugging-face')

login(token)

Cloning into 'diffusers'...
remote: Enumerating objects: 126308, done.
remote: Counting objects: 100% (425/425), done.
remote: Compressing objects: 100% (208/208), done.
remote: Total 126308 (delta 322), reused 218 (delta 217), pack-reused 125883 (from 3)
Receiving objects: 100% (126308/126308), 100.31 MiB | 21.98 MiB/s, done.
Resolving deltas: 100% (94070/94070), done.
/content/diffusers/examples/text_to_image


Para resolver o `ImportError`, precisamos garantir que a versão correta da biblioteca `diffusers` seja usada. Como o erro sugere uma instalação a partir do código-fonte, vamos desinstalar a versão instalada pelo `pip` e depois instalar a versão do repositório clonado no modo de desenvolvimento. Além disso, você precisará baixar seu dataset para `/content/dataset`.

In [2]:
# Desinstala a versão atual do diffusers instalada via pip
!pip uninstall -y diffusers

Found existing installation: diffusers 0.38.0
Uninstalling diffusers-0.38.0:
  Successfully uninstalled diffusers-0.38.0


In [3]:
# Instala o diffusers a partir do código-fonte clonado em modo editável
# Isso garantirá que a versão mais recente ou a versão do branch principal seja usada.
%cd /content/diffusers
!pip install -e .
%cd examples/text_to_image

/content/diffusers
Obtaining file:///content/diffusers
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for diffusers (pyproject.toml) ... done
  Created wheel for diffusers: filename=diffusers-0.40.0.dev0-0.editable-py3-none-any.whl size=11519 sha256=0f9f3e4d247ee04441bff9cbbe9d501b3f0b9eb2d7f9f69652c9c95da4eb10a2
  Stored in directory: /tmp/pip-ephem-wheel-cache-gt1kf5yy/wheels/8a/fc/09/385efb77b455b2fd4a656c950079c93147e1f50ae614e51beb
Successfully built diffusers
/content/diffusers/examples/text_to_image


In [4]:
from datasets import load_dataset
import pandas as pd
import os

# 1. Clone o repositório
!git clone https://github.com/hpsj2712/atelie-generativo-individual-homerio /content/temp_repo

# 2. Prepare o diretório do dataset
!mkdir -p /content/dataset

# 3. Mova as imagens para a pasta final
!mv /content/temp_repo/dados/*.jpg /content/dataset/

# 4. Processar as legendas (legendas.txt)
# O formato atual é "nome.jpg: legenda"
legendas_dict = {}
with open('/content/temp_repo/dados/legendas.txt', 'r', encoding='utf-8') as f:
    for line in f:
        if ":" in line:
            parts = line.split(":", 1)
            file_name = parts[0].strip()
            caption = parts[1].strip()
            legendas_dict[file_name] = caption

# 5. Criar o metadata.csv no formato exigido pelo Diffusers
df = pd.DataFrame(list(legendas_dict.items()), columns=['file_name', 'text'])
df.to_csv('/content/dataset/metadata.csv', index=False)

print("Dataset e metadata.csv criados com sucesso em /content/dataset/")
!rm -rf /content/temp_repo

Cloning into '/content/temp_repo'...
remote: Enumerating objects: 137, done.
remote: Counting objects: 100% (137/137), done.
remote: Compressing objects: 100% (118/118), done.
remote: Total 137 (delta 32), reused 31 (delta 10), pack-reused 0 (from 0)
Receiving objects: 100% (137/137), 128.11 MiB | 28.36 MiB/s, done.
Resolving deltas: 100% (32/32), done.
Dataset e metadata.csv criados com sucesso em /content/dataset/


Agora que a biblioteca `diffusers` foi atualizada a partir do código-fonte e o seu dataset está no lugar correto, você pode tentar executar novamente a célula `-Sf1Yi4BwTdv`.

In [5]:
!accelerate launch train_text_to_image_lora.py \
  --pretrained_model_name_or_path="stable-diffusion-v1-5/stable-diffusion-v1-5" \
  --train_data_dir="/content/dataset" \
  --resolution=512 \
  --train_batch_size=1 \
  --gradient_accumulation_steps=4 \
  --max_train_steps=1500 \
  --learning_rate=1e-4 \
  --lr_scheduler="cosine" \
  --rank=8 \
  --mixed_precision="fp16" \
  --checkpointing_steps=500 \
  --validation_prompt="estilo_arquitetura_modernista_brasilia, um predio oficial no inverno" \
  --output_dir="/content/lora_saida" \
  --push_to_hub \
  --hub_model_id="Junior/estilo_arquitetura_modernista_brasilia"

The following values were not passed to `accelerate launch` and had defaults used instead:
	`--num_processes` was set to a value of `0`
	`--num_machines` was set to a value of `1`
	`--mixed_precision` was set to a value of `'no'`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
INFO:__main__:[RANK 0] Distributed environment: DistributedType.NO
Num processes: 1
Process index: 0
Local process index: 0
Device: cpu

Mixed precision type: fp16

INFO:httpx:HTTP Request: POST https://huggingface.co/api/repos/create "HTTP/1.1 403 Forbidden"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/m